In [ ]:

#import all needed librarires
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

#by copy the path we will read the csv
csv_path = os.path.join(path, "/kaggle/input/q3-ka-ai-2026/Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:
#get all info
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# 2. Do we have missing values?
#Identify missing values and fill numerical columns using mean/median and categorical columns using mode or a constant value
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
# 4. Check for duplicates
#Remove duplicate rows to prevent biased learning and inflated performance metrics
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
#check if we have categorical columns, if so we will encode them, if not then we wont do anything. bcuz the questions aysy only if needed
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
#Scale only the input features to improve model performance and convergence while keeping the target variable unchanged.
#I NEED TO CHOOSE THE FEATURE SO I CAN SCALE IT

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("status")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# Task 5: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()


if (check_target_imbalance(df, "Target")== True):
  print("Imabalanced")
else:
  print("not imbalanced")

In [ ]:
# The softmax function
def softmax(z):
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
# Categorical cross entropy loss function using NumPy
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss

In [ ]:
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

In [ ]:
def gradient_descent(X, y, num_classes, lr, n_iters=1000):
  # Get the number of samples (m) and number of features (n)
  m, n = X.shape

  # Initialize weight matrix with shape (n, num_classes)
  theta = np.zeros((n, num_classes))

  # One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)

  losses = []

  for _ in tqdm(range(n_iters), desc="Training Multiclass Logistic Regression"):
    # Calculate the logits z
    z = np.dot(X, theta)

    # Get class probabilities using softmax
    y_pred = softmax(z)

    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m

    # Update weights
    theta -= lr * gradient

    # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)

  return theta, losses

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Task 1: Write your code here:
# Prepare the data as X and y

X = df.drop("Target", axis=1).astype(float)
y = df["Target"].astype(float)


In [ ]:

# Task 2,3,4,5: Write your code here:
n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

In [ ]:
#TODO: Calculate the average losses across folds
avg_loss = np.mean(sr_results['loss'], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("SOFTMAX REGRESSION Performance:")
# TODO: Print the average of each evaluation metric across folds
print(f"  Accuracy:  {np.mean(sr_results['acc']):.4f}")
print(f"  F1-Score:  {np.mean(sr_results['f1']):.4f}")

In [ ]:
!pip install xgboost catboost

In [ ]:
# Import models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
models = {
  "LR": LogisticRegression(
      max_iter=1000
  ),
  "KNN": KNeighborsClassifier(
      n_neighbors=3,
  ),
  "SVM": SVC(
      kernel='rbf',
      C=1.0
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=16
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=200,
      max_depth=10
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=280,
      max_depth=10,
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

In [ ]:
results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
for model_name in results:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': np.mean(np.abs(theta), axis=1)
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 16))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:

for model_name in results:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")

In [ ]:
# Task Bonus: Write your code here:
sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)
